# GameTheory-04d-Marchandage-Asymetrique

**Navigation** : [<< 4-NashEquilibrium](GameTheory-04-NashEquilibrium.ipynb) | [Index](README.md) | [4e-Reflective-Oracles >>](GameTheory-04e-Reflective-Oracles.ipynb)

## Marchandage asymétrique : dépendance, désir et point de désaccord

### Objectifs d'apprentissage

1. Formuler le point de désaccord (Nash, 1950) comme réponse à « que se passe-t-il si rien ne s'accorde ? ».
2. Distinguer le **désir** (une composante du faisceau) de la **dépendance** (le faisceau entier).
3. Construire un contre-exemple au principe du moindre intérêt et en tester la robustesse au générateur de poids.
4. Lire honnêtement la dissociation relationnelle : l'émancipation externe réduit le levier brut mais concentre l'asymétrie relationnelle.

### Prérequis
Équilibres de Nash purs et mixtes ([GT-04](GameTheory-04-NashEquilibrium.ipynb)) ; intuitions de base
sur les préférences et les alternatives extérieures (`CLalt`).


In [1]:
import numpy as np


## Le point de désaccord et le pouvoir : dépendance, désir et point de désaccord

L'équilibre de Nash dit quand plus rien ne bouge ; le **point de désaccord**
du marchandage (Nash, 1950) dit ce que chacun obtient si rien ne s'accorde.
Cette section relie ce point de désaccord à la question du **pouvoir dans une
relation** — le deuxième temps du « troisième voyage » de la série (après
l'humour, avant l'engagement, cf GameTheory-06c §7).

L'énoncé populaire — « celui qui désire le moins contrôle le plus » — est une
**loi psychologique fausse** telle quelle. Sa version sérieuse existe depuis
les années 1950, et le vocabulaire académique vaut mieux que l'intuitif :

| Notion | Nom canonique | Ce qu'elle dit exactement |
|--------|---------------|---------------------------|
| L'asymétrie de dépendance | **principe du moindre intérêt** (Waller, 1938) | celui dont l'intérêt au maintien de la relation est le plus faible dispose du levier |
| Sa forme relationnelle | **power-dependence** (Emerson, 1962) | le pouvoir de A sur B est la **dépendance de B envers A** — pas un trait de A |
| Sa forme psychologique | théorie de l'**interdépendance** (Thibaut & Kelley, 1959) | `CLalt` = qualité de la meilleure alternative disponible |
| Sa forme économique | **point de désaccord** (Nash, 1950) ; offres alternées (Rubinstein, 1982) | le partage dépend de ce que chacun obtient si l'accord échoue |
| Son maintien dans le temps | **modèle d'investissement** (Rusbult, 1980) | engagement ≈ satisfaction + investissements − alternatives |

Le gain de ce vocabulaire : il rend l'énoncé **faux de la bonne manière**. La
dépendance n'est pas un scalaire, c'est un **faisceau** — revenu, logement,
réseau, statut, affection. Le désir n'en est **qu'une composante**. On
construit un marchandage où le levier se calcule depuis le faisceau entier,
puis on cherche le **contre-exemple** : une instance où la partie qui désire
le **plus** l'emporte néanmoins, parce que sa dépendance sur les **autres**
composantes est plus faible.

In [2]:
# --- Marchandage a dependance multi-composantes (#12682) ---
# Chaque partie i porte un FAISCEAU de dependances. Pour chaque composante k :
#   gap_k = w_k * max(v_k - o_k, 0)   avec v_k = valeur DANS la relation,
#                                     o_k = meilleure option EXTERIEURE (CLalt locale)
# D_i = somme des gaps = ce que i perd en partant (point de desaccord pondere).
# Le "desir" scalaire = la composante affective SEULE du faisceau.

COMPOSANTES = ["affection", "revenu", "logement", "reseau", "statut"]

def faisceau_dependance(v, o, w):
    """Gap par composante et dependance totale D (Emerson 1962, operationalise)."""
    gaps = {k: w[k] * max(v[k] - o[k], 0.0) for k in COMPOSANTES}
    return gaps, sum(gaps.values())

def partage_nash(D_A, D_B, kappa=6.0, poids_constants=False):
    """Part du surplus relationnel capturee par A (marchandage de Nash asymetrique).

    HYPOTHESE DE MODELISATION (statut explicite) :
        Le poids de negociation de B suit une sigmoide du gap de dependance,
            xi_B = sigmoid(kappa * (D_A - D_B))
        avec kappa = 6.0 FIXE ARBITRAIREMENT. Ce n'est PAS une derivation de
        Nash 1950 ni de Rubinstein 1982 : c'est un POSTULAT de la famille
        "Nash asymetrique a poids endogenes", qui dit que la sensibilite d'une
        partie au cout de rupture croitra plus que lineaire avec le deficit.
        Le lien a Rubinstein est motivationnel, pas deductif : la dependance
        est lue comme un cout de rupture, mais la forme sigmoide n'est pas
        derivee. Lire la cellule suivante pour la sensibilite a kappa et le
        statut complet de cette hypothese.

    Si kappa = 0, partage_nash retourne 0.5 pour tout (D_A, D_B) : la forme
    perd toute information sur la dependance. Si kappa tend vers l'infini,
        la part tend vers un indicatrice de (D_A > D_B) : tout ou rien.
        Le renversement "A desire plus et A capture plus" tient des que
        D_B > D_A ET kappa > 0.
    poids_constants=True : controle degrade (#13313, reserve Hermes sur
    #12737). Le chemin endogene est debranche -- xi_B vaut exactement 0.5
    quel que soit l'ecart de dependance, la ligne de base a laquelle mesurer
    ce que le generateur de poids apporte.
    """
    if poids_constants:
        return 0.5, 0.5
    xi_B = 1.0 / (1.0 + np.exp(-kappa * (D_A - D_B)))
    return 1.0 - xi_B, xi_B

# Instance : A desire fort MAIS ses options exterieures sont confortables
# partout ailleurs ; B desire modement mais ses options exterieures externes
# (revenu, logement, statut) sont faibles.
v_A = {"affection": 0.9, "revenu": 0.8, "logement": 0.7, "reseau": 0.6, "statut": 0.6}
o_A = {"affection": 0.1, "revenu": 0.8, "logement": 0.7, "reseau": 0.6, "statut": 0.6}
v_B = {"affection": 0.6, "revenu": 0.7, "logement": 0.7, "reseau": 0.5, "statut": 0.5}
o_B = {"affection": 0.3, "revenu": 0.2, "logement": 0.1, "reseau": 0.5, "statut": 0.2}
w_A = {k: 0.2 for k in COMPOSANTES}   # poids uniformes, l'asymetrie est dans
w_B = {k: 0.2 for k in COMPOSANTES}   # les valeurs, pas dans l'echelle

gaps_A, D_A = faisceau_dependance(v_A, o_A, w_A)
gaps_B, D_B = faisceau_dependance(v_B, o_B, w_B)
desi_A, desi_B = gaps_A["affection"], gaps_B["affection"]

print("Faisceau de A (ce que A perd en partant) :")
for k in COMPOSANTES:
    print(f"  {k:>10} : {gaps_A[k]:.3f}")
print(f"  => D_A = {D_A:.3f}   (dont desir affectif : {desi_A:.3f})")
print()
print("Faisceau de B :")
for k in COMPOSANTES:
    print(f"  {k:>10} : {gaps_B[k]:.3f}")
print(f"  => D_B = {D_B:.3f}   (dont desir affectif : {desi_B:.3f})")

Faisceau de A (ce que A perd en partant) :
   affection : 0.160
      revenu : 0.000
    logement : 0.000
      reseau : 0.000
      statut : 0.000
  => D_A = 0.160   (dont desir affectif : 0.160)

Faisceau de B :
   affection : 0.060
      revenu : 0.100
    logement : 0.120
      reseau : 0.000
      statut : 0.060
  => D_B = 0.340   (dont desir affectif : 0.060)


In [3]:
# --- Contre-exemple au principe du moindre interet (livrable #12682) ---
part_A, xi_B = partage_nash(D_A, D_B)

print("Enonce populaire : \"celui qui desire le moins controle le plus\"")
print(f"desir affectif : desi_A = {desi_A:.3f}  >  desi_B = {desi_B:.3f}  (A desire PLUS)")
print(f"levier (part du surplus) : part_A = {part_A:.3f}  (B capte {1 - part_A:.3f})")
print()
if desi_A > desi_B and part_A > 0.5:
    print("CONTRE-EXEMPLE PRODUIT : la partie qui desire le plus l'emporte.")
    print("Le desir n'est qu'UNE composante du faisceau ; ici la dependance")
    print("externe de B (revenu + logement + statut) pese 0.280 contre 0.000")
    print("pour A, et c'est elle qui fait basculer le levier.")
else:
    print("l'instance ne produit pas le contre-exemple -- parametres a revoir")

Enonce populaire : "celui qui desire le moins controle le plus"
desir affectif : desi_A = 0.160  >  desi_B = 0.060  (A desire PLUS)
levier (part du surplus) : part_A = 0.746  (B capte 0.254)

CONTRE-EXEMPLE PRODUIT : la partie qui desire le plus l'emporte.
Le desir n'est qu'UNE composante du faisceau ; ici la dependance
externe de B (revenu + logement + statut) pese 0.280 contre 0.000
pour A, et c'est elle qui fait basculer le levier.


**Statut explicite de `partage_nash`.** La sigmoïde sur `D_A − D_B`
n'est **pas dérivée** d'un modèle de marchandage : c'est une **hypothèse de
modélisation** de la famille « Nash asymétrique à poids endogènes ». `kappa`
est un paramètre libre fixé à 6,0. Le calcul précédent montre le mécanisme ;
la cellule suivante teste si le **renversement** (A désire plus et A capture
plus) tient sur une plage de `kappa`, pas seulement à 6,0.

In [4]:
# --- Sensibilite a kappa : le renversement tient-il au-dela de 6.0 ? ---
# Le contre-exemple ci-dessus repose sur partage_nash(D_A, D_B, kappa=6).
# Question : si kappa change, le resultat "A desire plus et A capture plus"
# tient-il encore, ou depend-il du choix 6.0 ?
# Si oui pour tout kappa > 0, le renversement est une propriete du SIGNE
# de (D_B - D_A), pas du parametre -- c'est plus fort.

print(f"{'kappa':>8} | {'part_A':>7} | {'sign(D_B-D_A)>0':>16} | {'renversement':>13}")
print("-" * 55)
for k in [0.0, 1.0, 2.0, 4.0, 6.0, 8.0, 12.0, 20.0]:
    part_A_k, _ = partage_nash(D_A, D_B, kappa=k)
    renversement = "OUI" if (desi_A > desi_B and part_A_k > 0.5) else "non"
    print(f"{k:>8.1f} | {part_A_k:>7.3f} | {str(D_B > D_A):>16} | {renversement:>13}")

print()
print("Conclusion : le renversement est-il une propriete du signe de (D_B-D_A),")
print("ou depend-il du choix de kappa ?")
print(f"  D_B - D_A = {D_B - D_A:.3f}  (positif => B plus dependant => A capture plus)")
print(f"  Pour kappa -> 0+ : part_A tend vers 0.5 (indifferentiation).")
print(f"  Pour kappa -> +inf : part_A tend vers 1.0 (tout ou rien).")
print(f"  Le renversement tient pour tout kappa > 0 des que D_B > D_A : c'est")
print(f"  une propriete du SIGNE, pas de la valeur de kappa. La valeur exacte")
print(f"  de part_A est en revanche sensible au choix de kappa -- kappa=6")
print(f"  donne un levier «prononcé» (0.746), kappa=2 donne un levier modere")
print(f"  (proche de 0.5). Le MECANISME est robuste ; l'INTENSITE du levier,")
print(f"  non.")


   kappa |  part_A |  sign(D_B-D_A)>0 |  renversement
-------------------------------------------------------
     0.0 |   0.500 |             True |           non
     1.0 |   0.545 |             True |           OUI
     2.0 |   0.589 |             True |           OUI
     4.0 |   0.673 |             True |           OUI
     6.0 |   0.746 |             True |           OUI
     8.0 |   0.808 |             True |           OUI
    12.0 |   0.897 |             True |           OUI
    20.0 |   0.973 |             True |           OUI

Conclusion : le renversement est-il une propriete du signe de (D_B-D_A),
ou depend-il du choix de kappa ?
  D_B - D_A = 0.180  (positif => B plus dependant => A capture plus)
  Pour kappa -> 0+ : part_A tend vers 0.5 (indifferentiation).
  Pour kappa -> +inf : part_A tend vers 1.0 (tout ou rien).
  Le renversement tient pour tout kappa > 0 des que D_B > D_A : c'est
  une propriete du SIGNE, pas de la valeur de kappa. La valeur exacte
  de part_A est en 

**Lecture du contre-exemple.** A désire plus (0,160 contre 0,060 sur la
composante affective) et **l'emporte néanmoins** (74,6 % du surplus). La
dépendance totale de B envers la relation (0,340) dépasse celle de A (0,160)
parce que les options extérieures de B sont faibles sur les composantes
**externes** — revenu, logement, statut (0,280 cumulés, contre 0 pour A).

Le principe du moindre intérêt est ainsi **réfuté comme loi** : dire « le
désir commande » est faux dès que le faisceau contient autre chose que le
désir. Il est **conservé comme effet partiel** : à faisceau externe égal, la
composante affective ferait bien basculer le levier. C'est le bon statut
pour un énoncé populaire — ni vrai, ni inutile : vrai dans une tranche du
paramètre, faux comme universalité.

In [5]:
# --- Robustesse au generateur de poids (#13313, reserve Hermes sur #12737) ---
# La reserve : le poids de negociation est une SIGMOIDE de l'ecart de
# dependance -- le lien mesure passe peut-etre par un chemin que le
# modeleur a pose. Selon la forme fonctionnelle, le meme dispositif peut
# confirmer ou refuter le principe. On interroge donc le dispositif avec
# TROIS generateurs de poids :
#   (1) endogene  : xi_B = sigmoid(kappa * (D_A - D_B))  [famille #12737]
#   (2) exogene   : xi_B = z_B / (z_A + z_B)             [2e famille, ratio]
#   (3) constant  : xi_B = 0.5                           [controle degrade]
# Le clout exogene z_i = force des options exterieures (moyenne des o_k) :
# exogene a la relation, structurellement distinct de l'ecart de dependance.

def partage_nash_exogene(z_A, z_B):
    """2e famille de poids (#13313) : ratio de clout exogene, pas une
    fonction de l'ecart de dependance. Forme multiplicative la plus simple
    qui ne consulte JAMAIS D_A - D_B."""
    xi_B = z_B / (z_A + z_B)
    return 1.0 - xi_B, xi_B

z_A_exo = float(np.mean([o_A[k] for k in COMPOSANTES]))
z_B_exo = float(np.mean([o_B[k] for k in COMPOSANTES]))

part_endo, _ = partage_nash(D_A, D_B)
part_exo, _ = partage_nash_exogene(z_A_exo, z_B_exo)
part_const, _ = partage_nash(D_A, D_B, poids_constants=True)

print(f"Instance #12682 (desi_A = {desi_A:.3f} > desi_B = {desi_B:.3f}) :")
print(f"  (1) endogene, sigmoid(gap) : part_A = {part_endo:.3f}")
print(f"  (2) exogene, ratio z       : part_A = {part_exo:.3f}  (z_A = {z_A_exo:.2f}, z_B = {z_B_exo:.2f})")
print(f"  (3) constant, debranche    : part_A = {part_const:.3f}")

# Sensibilite : 200 instances perturbees (v/o bruites, tirage fixe).
# Taux de reproduction du contre-exemple (precondition desi_A > desi_B,
# puis part_A > 0.5) par generateur.
rng = np.random.default_rng(13313)
N = 200
n_pre = 0
repro = {"endogene": 0, "exogene": 0, "constant": 0}
for _ in range(N):
    v_A_p = {k: float(np.clip(v_A[k] + rng.normal(0, 0.08), 0, 1)) for k in COMPOSANTES}
    o_A_p = {k: float(np.clip(o_A[k] + rng.normal(0, 0.08), 0, 1)) for k in COMPOSANTES}
    v_B_p = {k: float(np.clip(v_B[k] + rng.normal(0, 0.08), 0, 1)) for k in COMPOSANTES}
    o_B_p = {k: float(np.clip(o_B[k] + rng.normal(0, 0.08), 0, 1)) for k in COMPOSANTES}
    gaps_Ap, D_Ap = faisceau_dependance(v_A_p, o_A_p, w_A)
    gaps_Bp, D_Bp = faisceau_dependance(v_B_p, o_B_p, w_B)
    if gaps_Ap["affection"] <= gaps_Bp["affection"]:
        continue
    n_pre += 1
    z_Ap = float(np.mean(list(o_A_p.values())))
    z_Bp = float(np.mean(list(o_B_p.values())))
    if partage_nash(D_Ap, D_Bp)[0] > 0.5:
        repro["endogene"] += 1
    if partage_nash_exogene(z_Ap, z_Bp)[0] > 0.5:
        repro["exogene"] += 1
    if partage_nash(D_Ap, D_Bp, poids_constants=True)[0] > 0.5:
        repro["constant"] += 1

print()
print(f"Sensibilite ({N} instances perturbees, precondition desi_A > desi_B atteinte {n_pre} fois) :")
for gen, label in (("endogene", "endogene, sigmoid(gap)"),
                   ("exogene", "exogene, ratio z"),
                   ("constant", "constant, debranche")):
    rate = 100.0 * repro[gen] / max(n_pre, 1)
    print(f"  {label:<22} : contre-exemple reproduit {repro[gen]}/{n_pre} ({rate:.0f}%)")

Instance #12682 (desi_A = 0.160 > desi_B = 0.060) :
  (1) endogene, sigmoid(gap) : part_A = 0.746
  (2) exogene, ratio z       : part_A = 0.683  (z_A = 0.56, z_B = 0.26)
  (3) constant, debranche    : part_A = 0.500

Sensibilite (200 instances perturbees, precondition desi_A > desi_B atteinte 199 fois) :
  endogene, sigmoid(gap) : contre-exemple reproduit 199/199 (100%)
  exogene, ratio z       : contre-exemple reproduit 199/199 (100%)
  constant, debranche    : contre-exemple reproduit 0/199 (0%)


In [6]:
# --- Couche #13313 : poids exogene parametre par la pente du dehors ---
# Le substrat ci-dessus (via #12737) fixe le clout exogene a la moyenne brute
# des options, soit une pente implicite gamma = 1. La couche generalise la
# pente : gamma joue pour le capital exterieur le role que kappa joue pour
# l'ecart endogene. Le dehors construit le clout, le substrat fait le ratio.
def partage_nash_dehors(o_A, o_B, gamma=2.0):
    """Poids exogene parametre : le dehors donne le pouvoir de menace.

    part_A = o_bar_A^gamma / (o_bar_A^gamma + o_bar_B^gamma), avec
    o_bar_i = moyenne des options exterieures de i. Ni l'ecart de
    dependance, ni les valeurs v_k n'entrent dans le poids : seul le
    dehors compte. Couche sur partage_nash_exogene (#12737) : la couche
    construit le clout z_i = o_bar_i ** gamma, le substrat fait le ratio.
    """
    o_bar_A = sum(o_A[k] for k in COMPOSANTES) / len(COMPOSANTES)
    o_bar_B = sum(o_B[k] for k in COMPOSANTES) / len(COMPOSANTES)
    z_A = o_bar_A ** gamma
    z_B = o_bar_B ** gamma
    return partage_nash_exogene(z_A, z_B)

# Equivalence gamma = 1 : la couche doit redonner exactement le ratio du
# substrat (clout bruts). C'est le pont verifiable entre les deux niveaux.
p_dehors_1, _ = partage_nash_dehors(o_A, o_B, gamma=1.0)
print(f"gamma = 1 : couche = {p_dehors_1:.3f} vs substrat = {part_exo:.3f}"
      f"  (equivalence exacte : {abs(p_dehors_1 - part_exo) < 1e-12})")


gamma = 1 : couche = 0.683 vs substrat = 0.683  (equivalence exacte : True)


In [7]:
# --- Distinction structurelle : les deux familles ne voient pas les memes choses ---
# Faire varier la valeur affective DANS la relation (v_affection de A).
# La famille endogene suit : le desir de A entre dans son faisceau, donc dans
# l'ecart qui pilote le poids. La famille exogene n'a pas ce canal : le poids
# ne voit que les options exterieures, qui ne bougent pas.
print(f"{'v_affection(A)':>14} | {'desi_A':>7} | {'part_A endo (k=6)':>18} | {'part_A exo (g=2)':>18}")
print("-" * 68)
for v_aff in [0.5, 0.7, 0.9, 1.0]:
    v_A_t = dict(v_A); v_A_t["affection"] = v_aff
    gaps_A_t, D_A_t = faisceau_dependance(v_A_t, o_A, w_A)
    p_endo, _ = partage_nash(D_A_t, D_B)
    p_exo, _ = partage_nash_dehors(o_A, o_B)
    print(f"{v_aff:>14.1f} | {gaps_A_t['affection']:>7.3f} | {p_endo:>18.3f} | {p_exo:>18.3f}")
print()
print("Deux lectures differentes du meme mouvement :")
print("1. endogene : le levier de A DECLINE quand son desir monte (0.826 ->")
print("   0.723) -- c'est la mecanique du moindre interet, intra-famille,")
print("   sur la seule composante ou A depend ;")
print("2. exogene : rien ne bouge (0.823 constant) -- le poids n'a pas ce canal.")
print("Les deux familles sont donc structurellement distinctes : changer de")
print("famille n'est pas changer un reglage, c'est changer ce que le poids voit.")

v_affection(A) |  desi_A |  part_A endo (k=6) |   part_A exo (g=2)
--------------------------------------------------------------------
           0.5 |   0.080 |              0.826 |              0.823
           0.7 |   0.120 |              0.789 |              0.823
           0.9 |   0.160 |              0.746 |              0.823
           1.0 |   0.180 |              0.723 |              0.823

Deux lectures differentes du meme mouvement :
1. endogene : le levier de A DECLINE quand son desir monte (0.826 ->
   0.723) -- c'est la mecanique du moindre interet, intra-famille,
   sur la seule composante ou A depend ;
2. exogene : rien ne bouge (0.823 constant) -- le poids n'a pas ce canal.
Les deux familles sont donc structurellement distinctes : changer de
famille n'est pas changer un reglage, c'est changer ce que le poids voit.


In [8]:
# --- Sensibilite a gamma : miroir exact de la sensibilite a kappa ---
# Meme question que pour kappa : la conclusion depend-elle de la pente
# choisie, ou seulement du signe de (o_bar_A - o_bar_B) ?
print(f"{'gamma':>8} | {'part_A exogene':>14} | {'contre-exemple':>14}")
print("-" * 44)
for g in [0.5, 1.0, 2.0, 4.0, 6.0, 8.0]:
    part_A_g, _ = partage_nash_dehors(o_A, o_B, gamma=g)
    print(f"{g:>8.1f} | {part_A_g:>14.3f} | {'OUI' if part_A_g > 0.5 else 'non':>14}")
print()
print(f"o_bar_A = {sum(o_A[k] for k in COMPOSANTES)/len(COMPOSANTES):.3f} > "
      f"o_bar_B = {sum(o_B[k] for k in COMPOSANTES)/len(COMPOSANTES):.3f}")
print("Comme pour kappa : la DIRECTION (part_A > 0.5) tient pour tout gamma > 0")
print("-- le signe de l'ecart de capital exterieur suffit ; l'INTENSITE va de")
print("0.595 (gamma=0.5) a 0.998 (gamma=8), exactement comme le levier endogene")
print("s'elargissait avec kappa. La pente est un choix de modelisation, pas un")
print("resultat.")

   gamma | part_A exogene | contre-exemple
--------------------------------------------
     0.5 |          0.595 |            OUI
     1.0 |          0.683 |            OUI
     2.0 |          0.823 |            OUI
     4.0 |          0.956 |            OUI
     6.0 |          0.990 |            OUI
     8.0 |          0.998 |            OUI

o_bar_A = 0.560 > o_bar_B = 0.260
Comme pour kappa : la DIRECTION (part_A > 0.5) tient pour tout gamma > 0
-- le signe de l'ecart de capital exterieur suffit ; l'INTENSITE va de
0.595 (gamma=0.5) a 0.998 (gamma=8), exactement comme le levier endogene
s'elargissait avec kappa. La pente est un choix de modelisation, pas un
resultat.


**Verdict de robustesse — ROBUSTE au générateur de poids.** Une objection méthodologique se pose : si le poids de négociation est lui-même une fonction de l'écart de dépendance, la réfutation du principe du moindre intérêt peut être tautologique — posée par le modélisateur plutôt que mesurée. L'interrogation à trois générateurs tranche : **la conclusion appartient au dispositif, pas à la forme fonctionnelle**.

| Générateur | Le poids dépend de | `part_A` | Contre-exemple |
|------|--------------------|----------|----------------|
| Endogène (κ=6) | écart `D_A − D_B` (construit du modèle) | 0,746 | OUI (199/199) |
| Exogène γ=1 (clout brut) | capital extérieur `ō` (options dehors) | 0,683 | OUI (199/199) |
| Exogène γ=2 (couche `partage_nash_dehors`) | capital extérieur `ō`, pente γ | 0,823 | OUI |
| Constants (contrôle) | rien — canal coupé | 0,500 | non (neutralité) |

1. **La réfutation comme loi est ROBUSTE (tous bras).** Le principe du moindre intérêt prédit `part_A < 0,5` quand `desi_A > desi_B`. Aucun bras ne le produit. Même le contrôle constants — qui coupe **tout** canal vers le poids — donne 0,500, pas la prédiction directionnelle du principe : sans information sur le faisceau, le désir seul ne prédit rien. La loi échoue pour une raison indépendante du générateur de poids.
2. **Le renversement positif (« A désire plus et l'emporte ») appartient au dispositif (2/2 familles informatives, 199/199 chacune).** Deux chemins structurellement distincts — l'écart de dépendance et le capital extérieur — donnent la même direction sur cette instance. Ce n'est pas tautologique : le différenciateur ci-dessus montre qu'ils ne réagissent pas aux mêmes variables.
3. **Borne 1 — l'intensité est propre à la famille et à sa pente.** L'écart à la ligne de base va de +0,183 (exogène γ=1) à +0,246 (endogène κ=6) et +0,323 (γ=2) ; le balayage γ ∈ [0,5 ; 8] donne 0,595 → 0,998, exactement comme le levier endogène s'élargissait avec κ. La **direction** est une propriété du dispositif, le **chiffre** une propriété de la fonction.
4. **Borne 2 — l'effet partiel est conditionnel à la famille endogène.** « À faisceau externe égal, le désir fait basculer le levier » n'existe que dans la famille où le poids voit le faisceau ; dans la famille exogène, le désir n'a aucun canal (différenciateur : 0,823 constant). La lecture précédente reste exacte **pour le modèle posé** ; généraliser l'effet partiel à toute famille de poids serait un artefact.

**Statut :** réfutation **robuste à la famille de poids** ; le chiffre 74,6 % et l'effet partiel sont des propriétés de la famille endogène, à lire comme telles.

In [9]:
# --- Bascule : emancipation externe -> l'asymetrie restante pese plus ---
# Les options exterieures de B s'ameliorent progressivement sur les
# composantes EXTERNES (revenu, logement, statut). L'affection, elle, ne
# change pas : elle est relationnelle, pas emancipable de l'exterieur.
print(f"{'etape':>6} | {'D_B':>6} | {'part_A':>7} | {'affect. dans D_B':>16} | {'affect. dans D_A':>16}")
for t in np.linspace(0.0, 1.0, 6):
    o_B_t = dict(o_B)
    for k in ("revenu", "logement", "statut"):
        o_B_t[k] = min(o_B[k] + t * (v_B[k] - o_B[k]), v_B[k])
    gaps_B_t, D_B_t = faisceau_dependance(v_B, o_B_t, w_B)
    part_A_t, _ = partage_nash(D_A, D_B_t)
    pa_B = gaps_B_t["affection"] / D_B_t if D_B_t > 1e-9 else float("nan")
    pa_A = gaps_A["affection"] / D_A
    print(f"{t:>6.1f} | {D_B_t:>6.3f} | {part_A_t:>7.3f} | {pa_B:>15.1%} | {pa_A:>15.1%}")

print()
print("Deux mouvements DISTINCTS se produisent en meme temps :")
print("1. le levier BRUT de A decline (0.746 -> 0.354) : la dependance externe")
print("   de B fond, l'ecart de dependance se retourne meme au-dela de t=0.8 ;")
print("2. la COMPOSITION de la dependance de B se concentre : l'affection passe")
print("   de 18% a 100% de son faisceau -- l'asymetrie restante est desormais")
print("   presque entierement relationnelle.")

 etape |    D_B |  part_A | affect. dans D_B | affect. dans D_A
   0.0 |  0.340 |   0.746 |           17.6% |          100.0%
   0.2 |  0.284 |   0.678 |           21.1% |          100.0%
   0.4 |  0.228 |   0.601 |           26.3% |          100.0%
   0.6 |  0.172 |   0.518 |           34.9% |          100.0%
   0.8 |  0.116 |   0.434 |           51.7% |          100.0%
   1.0 |  0.060 |   0.354 |          100.0% |          100.0%

Deux mouvements DISTINCTS se produisent en meme temps :
1. le levier BRUT de A decline (0.746 -> 0.354) : la dependance externe
   de B fond, l'ecart de dependance se retourne meme au-dela de t=0.8 ;
2. la COMPOSITION de la dependance de B se concentre : l'affection passe
   de 18% a 100% de son faisceau -- l'asymetrie restante est desormais
   presque entierement relationnelle.


**Lecture de la bascule — la dissociation historique.** Émanciper des
dépendances **externes** (revenu, statut, autonomie juridique et résidentielle)
est un progrès ; l'effet mécanique dans ce modèle est double : le levier brut
de l'autre partie **décline** (0,746 → 0,354) — et la part **relative** de
l'asymétrie relationnelle restante **monte** (18 % → 100 % de la dépendance de
B). L'asymétrie restante ne devient pas plus grande en absolu : elle devient
presque **toute** l'asymétrie visible. C'est la formulation exacte de la
dissociation :

```
émancipation des dépendances externes  ≠>  disparition des asymétries relationnelles
```

Le modèle ne dit pas qu'il faudrait regretter la contrainte ancienne — il dit
que la négociation change de **nature** : moins d'enjeux de survie, plus
d'enjeux relationnels purs. Rien dans ce résultat ne peut servir d'argument
contre l'émancipation.

**Deux questions ouvertes — la part qui pose des questions, pas des variables**

La psychanalyse et la sociologie compréhensive entrent ici comme
**productrices de distinctions**, jamais de paramètres. Une question écrite,
suivie de ce qu'il faudrait mesurer pour y répondre — et d'aucun coefficient.

> **1. Désir mimétique (Girard, 1961).** La valeur d'un objet dépend du fait
> qu'un tiers le désire. Le modèle ci-dessus traite `o_k` et `v_k` comme
> exogènes. *Que faudrait-il ajouter pour que la valeur de la relation devienne
> endogène au désir d'autrui — et qu'est-ce qui, dans une trajectoire observée,
> permettrait de distinguer un désir de l'objet d'un désir par l'intermédiaire
> de l'objet ?*
>
> Ce qu'il faudrait mesurer : une variation du « désirabilité » de la relation
> **indépendante de ses propriétés intrinsèques** — par exemple une variation
> de sa visibilité sociale auprès de tiers — et l'effet de cette variation sur
> `v_affection` estimé avant/après. Sans cette exogénéité contrôlée, mimétisme
> et préférence stable sont indiscernables.

> **2. Demande versus désir (Lacan) / espace transitionnel (Winnicott).** La
> demande est formulable et négociable ; le désir reste au-delà. Le modèle
> réduit tout le faisceau à des grandeurs négociables. *Quelle observabilité
> faudrait-il pour distinguer, dans le comportement de négociation d'un agent,
> ce qui relève de la demande satisfaite de ce qui relève du désir maintenu —
> sachant qu'un agent peut refuser un accord avantageux précément pour
> préserver le désir ?*
>
> Ce qu'il faudrait mesurer : des **refus d'accords dominants** — des
> trajectoires où l'agent décline une part de surplus supérieure à son point de
> désaccord. Dans le modèle, ce comportement est une erreur ; hors du modèle,
> c'est un signal. La fréquence et la sélectivité de ces refus (sur quelles
> composantes ?) seraient l'observable.

La forme interdite, pour mémoire : `coefficient_mimetique = 0.3` — un
paramètre qui prétend mesurer ce qu'il vient d'illustrer ne mesure rien.

**Limites et anti-claims.** Ce marchandage est un **modèle partiel** du
désir, explicitement — jamais sa réduction. La négociation est ici un des
régimes possibles de la relation, pas son essence ; l'utilité n'est pas le
désir ; l'amour n'est pas une symétrie de dépendances. Aucune inférence d'état
mental attribué à une personne réelle : on étudie des **structures
d'incitations**, pas des personnes. Les composantes (affection, revenu,
logement, réseau, statut) sont des dimensions d'analyse sociologique, pas un
inventaire de la vie relationnelle — et le modèle d'investissement de Rusbult
(1980), dont ce faisceau est un parent pauvre, l'a établi empiriquement bien
avant nous.

**Références (vérifiées à l'écriture)** : Waller, W. (1938), *The Family: A
Dynamic Interpretation* ; Emerson, R. M. (1962), « Power-Dependence
Relations », *American Sociological Review* 27(1) ; Thibaut, J. W. & Kelley,
H. H. (1959), *The Social Psychology of Groups* ; Nash, J. F. (1950), « The
Bargaining Problem », *Econometrica* 18(2) ; Rubinstein, A. (1982), « Perfect
Equilibrium in a Bargaining Model », *Econometrica* 50(1) ; Rusbult, C. E.
(1980), « Commitment and satisfaction in romantic associations », *Journal of
Experimental Social Psychology* 16(2) ; Girard, R. (1961), *Mensonge
romantique et vérité romanesque*.

La robustesse de la réfutation au générateur de poids (sigmoïde endogène
vs ratio exogène vs contrôle constant) est établie dans la cellule de verdict
ci-dessus — verdict **ROBUSTE**.


In [10]:
# Exercice a completer
# Objectif : mesurer comment le levier de A repond a la valeur affective
# de la relation, puis interpreter le sens de la variation (moindre interet).

def exercice_levier_affection(v_aff_liste) -> dict:
    """Pour chaque valeur de v_A['affection'], calcule la part du surplus
    capturee par A (part_A, kappa=6). Les autres composantes de A restent a v_A.

    Indice : recopier le motif de la cellule 37 (boucle sur v_aff, varier
    v_A_t['affection'], puis faisceau_dependance(v_A_t, o_A, w_A) et
    partage_nash(D_t, D_B, kappa=6.0)).

    Args:
        v_aff_liste: liste de valeurs pour la composante affective de A.

    Returns:
        dict avec {'affection': v_aff_liste, 'part_A': [part_A pour chaque valeur]}
    """
    resultat = None  # TODO etudiant
    return resultat

test = exercice_levier_affection([0.5, 0.7, 0.9])
print("Reponse attendue (a completer) :")

# Indice : la cellule 37 montre que le levier de A DECLINE quand son desir
# monte -- c'est la mecanique intra-famille du principe du moindre interet.
# A tester ci-dessous une fois la fonction completee :
if test is not None:
    monos = all(test['part_A'][i] < test['part_A'][i+1] for i in range(len(test['part_A'])-1))
    print(f"  affection = {test['affection']}")
    print(f"  part_A    = {test['part_A']}")
    print(f"  monotone decroissante = {monos} (True attendu : moindre interet)")


Reponse attendue (a completer) :


**Retour** : [<< GT-04 Nash](GameTheory-04-NashEquilibrium.ipynb) | [Index GameTheory](README.md)
